# LayoutLMv3 SROIE — Final Test Recovery Acc 2 v1

Notebook này chỉ tiếp tục execution record đã được v3 tạo nhưng đã dừng **trước cell đọc test**. Nó không tạo run mới. Recovery sẽ bị chặn nếu final report đã tồn tại, execution không ở trạng thái `started`, hoặc lock đã có `test_access_started_at`.

Đây là recovery một lần cho sự cố tải processor ngày 13/10/2026. Không dùng notebook này cho một final-test run khác.


In [ ]:
from pathlib import Path

PROJECT_NAME = "AI_Project_Insurance_Acc 2"
CANDIDATE_REPORT_REL = Path("reports/metrics/documents/layoutlmv3_sroie_candidate_validation_v2.json")
SPLIT_REPORT_REL = Path("reports/data_audit/documents/sroie_processed_split_v1.json")
CHECKPOINT_REL = Path("artifacts/document/layoutlmv3_sroie_candidate_v2/best")
RUN_FINAL_TEST = True
EXPECTED_TEST_ROWS = 361
SEED = 20261012


In [ ]:
import subprocess, sys, importlib.util
requirements={"transformers":"transformers>=4.45,<6","accelerate":"accelerate>=1,<2","pyarrow":"pyarrow>=17,<26","PIL":"Pillow>=10,<13"}
for module,requirement in requirements.items():
    if importlib.util.find_spec(module) is None:
        subprocess.run([sys.executable,"-m","pip","install",requirement],check=True)
from google.colab import drive
drive.mount("/content/drive")

import hashlib, json, uuid
from datetime import datetime, timezone

project_candidates = [Path("/content/drive/MyDrive") / PROJECT_NAME]
shortcut_root = Path("/content/drive/.shortcut-targets-by-id")
if shortcut_root.is_dir():
    project_candidates.extend(shortcut_root.glob(f"*/{PROJECT_NAME}"))
project_candidates = list(dict.fromkeys(path.resolve() for path in project_candidates if path.is_dir()))
matching_projects = [path for path in project_candidates if (path / CANDIDATE_REPORT_REL).is_file()]
if not matching_projects:
    checked = ", ".join(str(path) for path in project_candidates) or "Không thấy project/shortcut nào"
    raise FileNotFoundError(
        "Không tìm thấy candidate report trong Drive của tài khoản hiện tại. "
        "Hãy mở notebook bằng tài khoản đã train LayoutLMv3 hoặc share toàn bộ project từ tài khoản đó. "
        f"Đã kiểm tra: {checked}. File cần có: {CANDIDATE_REPORT_REL}"
    )
PROJECT = matching_projects[0]
DATA_ROOT = PROJECT / "data/processed/documents/sroie/v1"
CANDIDATE_REPORT = PROJECT / CANDIDATE_REPORT_REL
SPLIT_REPORT = PROJECT / SPLIT_REPORT_REL
FINAL_REPORT = PROJECT / "reports/metrics/documents/layoutlmv3_sroie_final_test_v1.json"
EXECUTION_LOCK = PROJECT / "reports/metrics/documents/layoutlmv3_sroie_final_test_v1.execution.json"
print("Resolved project:", PROJECT)
print("Candidate report:", CANDIDATE_REPORT)

def sha256_file(path):
    digest=hashlib.sha256()
    with Path(path).open("rb") as stream:
        for chunk in iter(lambda:stream.read(1024*1024),b""): digest.update(chunk)
    return digest.hexdigest()

def tree_hash(root):
    digest=hashlib.sha256()
    for path in sorted(Path(root).rglob("*")):
        if path.is_file():
            digest.update(str(path.relative_to(root)).encode())
            digest.update(bytes.fromhex(sha256_file(path)))
    return digest.hexdigest()

if not RUN_FINAL_TEST: raise RuntimeError("RUN_FINAL_TEST chưa được mở.")
if FINAL_REPORT.exists(): raise FileExistsError(f"Final report đã tồn tại; không recovery: {FINAL_REPORT}")
if not EXECUTION_LOCK.exists(): raise FileNotFoundError(f"Không có execution lock để recovery: {EXECUTION_LOCK}")

selection=json.loads(CANDIDATE_REPORT.read_text(encoding="utf-8"))
split_report=json.loads(SPLIT_REPORT.read_text(encoding="utf-8"))
execution=json.loads(EXECUTION_LOCK.read_text(encoding="utf-8"))
if not selection.get("candidate_selected"): raise RuntimeError("Candidate chưa được chọn theo locked rule.")
if not selection.get("final_test_locked") or selection.get("test_used"): raise RuntimeError("Selection protocol không hợp lệ.")
if not split_report.get("valid") or any(split_report["cross_split_exact_duplicates"].values()): raise RuntimeError("Processed split chưa sạch.")
if execution.get("status") != "started" or execution.get("test_run_number") != 1: raise RuntimeError(f"Execution state không recovery được: {execution}")
if execution.get("test_access_started_at"): raise RuntimeError("Lock cho biết test access đã bắt đầu; không được đọc test lại.")

reported_checkpoint=Path(selection["checkpoint"])
CHECKPOINT=reported_checkpoint if reported_checkpoint.is_dir() else PROJECT/CHECKPOINT_REL
if not CHECKPOINT.is_dir(): raise FileNotFoundError(f"Thiếu checkpoint: {CHECKPOINT}")
checkpoint_hash=tree_hash(CHECKPOINT)
selection_hash=sha256_file(CANDIDATE_REPORT)
if checkpoint_hash != selection["checkpoint_tree_sha256"]: raise RuntimeError("Checkpoint tree hash không khớp selection report.")
if checkpoint_hash != execution.get("checkpoint_tree_sha256"): raise RuntimeError("Checkpoint tree hash không khớp execution lock.")
if selection_hash != execution.get("selection_report_sha256"): raise RuntimeError("Selection report hash không khớp execution lock.")
run_id=execution["run_id"]
print("Recovery gate: PASS")
print("Reusing run_id:",run_id)
print("Test access recorded previously: False")


## Định nghĩa dataset và metric đã khóa từ candidate


In [ ]:
import io, re, unicodedata
from collections import Counter
from datetime import date
from decimal import Decimal, InvalidOperation
from difflib import SequenceMatcher
import numpy as np
import pyarrow.parquet as pq
import torch
from PIL import Image
from torch.utils.data import Dataset
from transformers import AutoProcessor, LayoutLMv3ForTokenClassification, Trainer, TrainingArguments, set_seed

set_seed(SEED)
FIELD_MAP = {"date":"DATE","total":"TOTAL"}
LABELS = ["O"] + [prefix+"-"+name for name in FIELD_MAP.values() for prefix in ("B","I")]
LABEL2ID = {label:index for index,label in enumerate(LABELS)}
ID2LABEL = {index:label for label,index in LABEL2ID.items()}

def entity_text(value):
    if value is None: return ""
    if isinstance(value,str): return value.strip()
    if isinstance(value,dict):
        for key in ("text","value","answer"):
            if value.get(key): return entity_text(value[key])
        return " ".join(entity_text(item) for item in value.values() if item)
    if isinstance(value,(list,tuple,set)): return " ".join(entity_text(item) for item in value if item)
    return str(value).strip()

def normalize(value):
    text=unicodedata.normalize("NFKC",entity_text(value)).casefold()
    return "".join(character for character in text if character.isalnum())

def parsed_dates(value):
    text=unicodedata.normalize("NFKC",entity_text(value)); candidates=set()
    for first,second,third in re.findall(r"(?<!\d)(\d{1,4})[./-](\d{1,2})[./-](\d{1,4})(?!\d)",text):
        candidates.add((int(first),int(second),int(third)) if len(first)==4 else (int(third),int(second),int(first)))
    for digits in re.findall(r"(?<!\d)(\d{8})(?!\d)",text):
        candidates.add((int(digits[:4]),int(digits[4:6]),int(digits[6:])) if digits[:2] in {"19","20"} else (int(digits[4:]),int(digits[2:4]),int(digits[:2])))
    result=set()
    for year,month,day in candidates:
        try: result.add(date(year,month,day))
        except ValueError: pass
    return result

def parsed_amounts(value):
    text=unicodedata.normalize("NFKC",entity_text(value)); result=set()
    for candidate in re.findall(r"(?<!\d)[+-]?\d[\d., ]*(?!\d)",text):
        token=candidate.replace(" ","").strip(".,")
        if not token: continue
        if "," in token and "." in token:
            token=token.replace(",","") if token.rfind(".")>token.rfind(",") else token.replace(".","").replace(",",".")
        elif "," in token:
            tail=token.rsplit(",",1)[-1]; token=token.replace(",",".") if len(tail) in (1,2) else token.replace(",","")
        try: result.add(Decimal(token))
        except InvalidOperation: pass
    return result

def candidate_score(field,target,candidate):
    wanted,observed=normalize(target),normalize(candidate)
    if not wanted or not observed: return 0.0,"empty"
    if wanted==observed: return 1.0,"exact_compact"
    if field=="date":
        if parsed_dates(target) & parsed_dates(candidate): return 0.995,"equal_date"
        wanted_digits="".join(re.findall(r"\d",entity_text(target)))
        observed_digits="".join(re.findall(r"\d",entity_text(candidate)))
        if len(wanted_digits)>=6 and wanted_digits in observed_digits: return 0.99,"date_digits"
    if field=="total" and parsed_amounts(target) & parsed_amounts(candidate): return 0.99,"equal_amount"
    if wanted in observed: return 0.96,"target_contained"
    if observed in wanted and len(observed)/len(wanted)>=0.75: return 0.93,"candidate_contained"
    return SequenceMatcher(None,wanted,observed).ratio(),"fuzzy"

def align_entity(words,field,value,max_window=3,minimum_score=0.84):
    target=entity_text(value)
    if not target: return None
    ranked=[]; context_words=("date","dated","tarikh") if field=="date" else ("total","amount","grandtotal","jumlah")
    for start in range(len(words)):
        for end in range(start+1,min(len(words),start+max_window)+1):
            candidate=" ".join(str(word) for word in words[start:end])
            base,method=candidate_score(field,target,candidate)
            if base<minimum_score: continue
            context=normalize(" ".join(words[max(0,start-2):start+1]))
            rank=base+(0.20 if any(label in context for label in context_words) else 0.0)
            ranked.append((rank,base,-(end-start),start,end,method,candidate))
    if not ranked: return None
    _,score,_,start,end,method,candidate=max(ranked)
    return {"field":field,"start":start,"end":end,"score":score,"method":method,"target":target,"matched_text":candidate}

def weak_bio(words,entities,return_alignments=False):
    labels=["O"]*len(words); entities=entities if isinstance(entities,dict) else {}; alignments={}; occupied=set()
    for source,target in FIELD_MAP.items():
        match=align_entity(words,source,entities.get(source)); alignments[source]=match
        if match is None or any(index in occupied for index in range(match["start"],match["end"])): continue
        labels[match["start"]]="B-"+target
        for index in range(match["start"]+1,match["end"]): labels[index]="I-"+target
        occupied.update(range(match["start"],match["end"]))
    return (labels,alignments) if return_alignments else labels

BASE_MODEL = "microsoft/layoutlmv3-base"
BASE_REVISION = "4dcbeabd3db1a13c8aeead201bb7a4811904f1fd"
processor = AutoProcessor.from_pretrained(BASE_MODEL, revision=BASE_REVISION, apply_ocr=False)

class SROIE(Dataset):
    def __init__(self, path): self.rows = pq.read_table(path).to_pylist()
    def __len__(self): return len(self.rows)
    def row_values(self, index):
        row = self.rows[index]
        words = [str(word) for word in row["words"]]
        value = row["image"]; raw = value.get("bytes") if isinstance(value,dict) else value
        image = Image.open(io.BytesIO(raw)).convert("RGB")
        boxes = row.get("bboxes") or row.get("boxes")
        if boxes is None: raise RuntimeError(f"Missing bboxes. Columns: {sorted(row)}")
        width,height = image.size
        boxes = [[max(0,min(1000,round(1000*x1/width))),max(0,min(1000,round(1000*y1/height))),max(0,min(1000,round(1000*x2/width))),max(0,min(1000,round(1000*y2/height)))] for x1,y1,x2,y2 in boxes]
        word_labels = [LABEL2ID[label] for label in weak_bio(words,row.get("entities",{}))]
        return row, words, image, boxes, word_labels
    def __getitem__(self, index):
        row, words, image, boxes, word_labels = self.row_values(index)
        encoded = processor(image,words,boxes=boxes,word_labels=word_labels,truncation=True,padding="max_length",max_length=512,return_tensors="pt")
        return {key:value.squeeze(0) for key,value in encoded.items()}
    def evaluation_metadata(self):
        metadata = []
        for index in range(len(self)):
            row, words, _, boxes, word_labels = self.row_values(index)
            tokenized = processor.tokenizer(words,boxes=boxes,word_labels=word_labels,truncation=True,padding="max_length",max_length=512)
            metadata.append({"words":words,"entities":row.get("entities",{}),"word_ids":tokenized.word_ids()})
        return metadata


## Mở test và đánh giá đúng một lần


In [ ]:
# Atomic audit marker: from this line onward the locked test is considered opened.
execution.update({
    "status":"test_access_started",
    "test_access_started_at":datetime.now(timezone.utc).isoformat(),
    "recovery_notebook":"Document_LayoutLMv3_Final_Test_Recovery_Acc2_v1.ipynb",
})
EXECUTION_LOCK.write_text(json.dumps(execution,ensure_ascii=False,indent=2),encoding="utf-8")
print("Test-access marker written. Do not rerun if execution stops after this line.")

test_data = SROIE(DATA_ROOT/"test.parquet")
if len(test_data) != EXPECTED_TEST_ROWS: raise RuntimeError(f"Unexpected test rows: {len(test_data)}")
test_metadata = test_data.evaluation_metadata()
print("Final test rows:",len(test_data))

def metric_value(field,value):
    if field=="date" and parsed_dates(value): return min(parsed_dates(value)).isoformat()
    if field=="total" and parsed_amounts(value): return str(min(parsed_amounts(value)).normalize())
    return normalize(value)

def bio_spans(labels):
    spans = []
    active_name, active_start = None, None
    for index, label in enumerate(labels + ["O"]):
        prefix, name = (label.split("-",1) + [None])[:2] if "-" in label else (label, None)
        if active_name is not None and (prefix != "I" or name != active_name):
            spans.append((active_name, active_start, index))
            active_name, active_start = None, None
        if prefix == "B":
            active_name, active_start = name, index
        elif prefix == "I" and active_name is None:
            active_name, active_start = name, index
    return spans

def compute_metrics(result):
    predictions = np.argmax(result.predictions,axis=-1)
    true_spans, predicted_spans = Counter(), Counter()
    expected_fields, predicted_fields = Counter(), Counter()
    for sequence_index,(pred_row,label_row,metadata) in enumerate(zip(predictions,result.label_ids,test_metadata)):
        keep = label_row != -100
        truth = [ID2LABEL[int(x)] for x in label_row[keep]]
        predicted = [ID2LABEL[int(x)] for x in pred_row[keep]]
        true_spans.update((sequence_index,*span) for span in bio_spans(truth))
        predicted_spans.update((sequence_index,*span) for span in bio_spans(predicted))
        word_labels = ["O"] * len(metadata["words"])
        seen_word_ids = set()
        for token_index, word_id in enumerate(metadata["word_ids"]):
            if word_id is None or word_id in seen_word_ids: continue
            seen_word_ids.add(word_id)
            if word_id < len(word_labels): word_labels[word_id] = ID2LABEL[int(pred_row[token_index])]
        for entity_name,start,end in bio_spans(word_labels):
            field_name = {"COMPANY":"company","DATE":"date","ADDRESS":"address","TOTAL":"total"}[entity_name]
            predicted_fields[(sequence_index,field_name,metric_value(field_name," ".join(metadata["words"][start:end])))] += 1
        entities = metadata["entities"] if isinstance(metadata["entities"],dict) else {}
        for field_name in FIELD_MAP:
            if entities.get(field_name): expected_fields[(sequence_index,field_name,metric_value(field_name,entities[field_name]))] += 1
    true_positive = sum((true_spans & predicted_spans).values())
    precision = true_positive / sum(predicted_spans.values()) if predicted_spans else 0.0
    recall = true_positive / sum(true_spans.values()) if true_spans else 0.0
    f1 = 2*precision*recall/(precision+recall) if precision+recall else 0.0
    def field_scores(names):
        expected = Counter({key:value for key,value in expected_fields.items() if key[1] in names})
        predicted = Counter({key:value for key,value in predicted_fields.items() if key[1] in names})
        tp = sum((expected & predicted).values())
        p = tp/sum(predicted.values()) if predicted else 0.0
        r = tp/sum(expected.values()) if expected else 0.0
        return p,r,2*p*r/(p+r) if p+r else 0.0
    field_precision,field_recall,field_f1 = field_scores({"date","total"})
    date_precision,date_recall,date_f1 = field_scores({"date"})
    total_precision,total_recall,total_f1 = field_scores({"total"})
    all_precision,all_recall,all_f1 = field_scores(set(FIELD_MAP))
    return {"bio_span_precision":precision,"bio_span_recall":recall,"bio_span_f1":f1,"field_precision":field_precision,"field_recall":field_recall,"field_micro_f1":field_f1,"date_precision":date_precision,"date_recall":date_recall,"date_f1":date_f1,"total_precision":total_precision,"total_recall":total_recall,"total_f1":total_f1,"all_field_micro_f1":all_f1}

model = LayoutLMv3ForTokenClassification.from_pretrained(CHECKPOINT)
args = TrainingArguments(output_dir="/content/layoutlmv3_final_test_runtime",seed=SEED,per_device_eval_batch_size=2,report_to="none",fp16=torch.cuda.is_available())
trainer = Trainer(model=model,args=args,eval_dataset=test_data,processing_class=processor,compute_metrics=compute_metrics)
metrics = trainer.evaluate()
print(metrics)


## Ghi final report và đóng execution lock


In [ ]:
report={
 "created_at":datetime.now(timezone.utc).isoformat(),
 "run_id":run_id,"test_run_number":1,"split":"test","test_rows":len(test_data),
 "selected_checkpoint":str(CHECKPOINT),"checkpoint_tree_sha256":checkpoint_hash,
 "selection_report":str(CANDIDATE_REPORT),"selection_report_sha256":sha256_file(CANDIDATE_REPORT),
 "test_used_for_selection":False,
 "metrics":{key:float(value) for key,value in metrics.items() if isinstance(value,(int,float))},
 "scope":{"fields":["date","total"],"dataset":"SROIE processed v1"},
 "limitations":["Public receipt data is not Vietnamese insurance-domain validation.","Only date and total are evaluated."],
}
FINAL_REPORT.write_text(json.dumps(report,ensure_ascii=False,indent=2),encoding="utf-8")
execution.update({"status":"completed","completed_at":datetime.now(timezone.utc).isoformat(),"final_report":str(FINAL_REPORT),"final_report_sha256":sha256_file(FINAL_REPORT)})
EXECUTION_LOCK.write_text(json.dumps(execution,ensure_ascii=False,indent=2),encoding="utf-8")
print("=== LAYOUTLMV3 FINAL TEST: PASS ===")
print("Field micro-F1:",report["metrics"].get("eval_field_micro_f1"))
print("Date F1:",report["metrics"].get("eval_date_f1"))
print("Total F1:",report["metrics"].get("eval_total_f1"))
print("Final report:",FINAL_REPORT)
print("Không chạy lại final test.")
